# Anomaly Model Evaluation

Computes precision / recall / F1 for the Isolation Forest + rules layer by
comparing the model's `is_anomaly` flag against the producer's ground-truth
`is_anomaly_truth` label stored in ClickHouse.

This is what turns the project's earlier 'no formal evaluation' limitation
into a real, defensible metric.

In [ ]:
import clickhouse_connect
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

client = clickhouse_connect.get_client(host='localhost', port=8123,
                                       username='admin', password='admin123', database='pipeline')

In [ ]:
df = client.query_df('''
    SELECT is_anomaly, is_anomaly_truth, anomaly_kind_truth, anomaly_score
    FROM pipeline.events
    WHERE event_ts >= now() - INTERVAL 24 HOUR
''')
print(f'{len(df):,} events in the last 24h')
df.head()

In [ ]:
y_true = df['is_anomaly_truth']
y_pred = df['is_anomaly']

print('Precision:', round(precision_score(y_true, y_pred), 4))
print('Recall:   ', round(recall_score(y_true, y_pred), 4))
print('F1:       ', round(f1_score(y_true, y_pred), 4))
print()
print(classification_report(y_true, y_pred, target_names=['normal','anomaly']))

In [ ]:
# Per-anomaly-kind recall: which fraud types do we catch best?
anom = df[df['is_anomaly_truth'] == 1]
recall_by_kind = anom.groupby('anomaly_kind_truth')['is_anomaly'].mean().round(3)
recall_by_kind.sort_values(ascending=False)

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(cm, index=['actual_normal','actual_anomaly'],
             columns=['pred_normal','pred_anomaly'])